# DiffSinger Miadu Fine-tuning (全自動路徑偵測版)
**已修正**：
- **路徑自動搜索**：自動尋找雲端硬碟中的 `DiffSinger_Colab` 資料夾（無視空格）。
- **搬運校驗**：確保 `phone_set.json` 等關鍵檔案正確進入 `data/binary`。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：環境配置 (Python 3.12 補丁)

In [ ]:
%cd /content/
!git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
%cd diffsinger-repo

# 強行升級工具與安裝依賴 (避開 webrtcvad)
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第三步：全自動偵測 Drive 並搬運數據

In [ ]:
import os
import glob

# 搜索 Google Drive 根目錄，尋找包含 DiffSinger 的資料夾
search_path = "/content/drive/MyDrive/DiffSinger*Colab*"
found_paths = glob.glob(search_path)

if not found_paths:
    print("| [錯誤] 找不到您的雲端資料夾！請確保資料夾名稱包含 DiffSinger_Colab")
    drive_root = "/content/drive/MyDrive/DiffSinger_Colab" # 預設 fallback
else:
    drive_root = found_paths[0]
    print(f"| [成功] 偵測到雲端路徑: {drive_root}")

# 1. 軟連結 checkpoints
!mkdir -p "{drive_root}/checkpoints_finetune"
!rm -rf checkpoints
!ln -s "{drive_root}/checkpoints_finetune" checkpoints

# 2. 搬運預訓練權重
if not os.path.exists("checkpoints/1117_opencpop_ds1000_strict_pinyin"):
    !cp -r "{drive_root}/1117_opencpop_ds1000_strict_pinyin" checkpoints/
    !cp -r "{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02" checkpoints/

# 3. 搬運二進位數據 (確保 miadu_finetune 資料夾能對接)
!mkdir -p data/binary
if not os.path.exists("data/binary/miadu_finetune"):
    # 優先從 drive_root 尋找 miadu_finetune
    src_data = os.path.join(drive_root, "miadu_finetune")
    if os.path.exists(src_data):
        !cp -r "{src_data}" data/binary/
    else:
        print(f"| [警告] 找不到 {src_data}，請檢查上傳資料夾名稱")

# 4. 驗證關鍵檔案
if os.path.exists("data/binary/miadu_finetune/phone_set.json"):
    print("| [成功] phone_set.json 已定位，可以開始訓練。")
else:
    print("| [失敗] data/binary/miadu_finetune/phone_set.json 仍缺失，請手動確認目錄結構")

## 第四步：啟動訓練

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1